[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iitm-da/da2402/blob/master/endsem/practice/endsem_practice.ipynb)

# Endsem practice paper

DA2402 · Data Curation and Visualization · Dr. Arun B Ayyar

Thirteen problems across the whole course, in the order it was taught: pandas, regular expressions,
scraping, JSON, missing values, imputation, outliers, duplicates, storage, Dash and honest charts.
Nothing here is marked. Each problem names a variable `q1` … `q13`. Put your result in it, run the
cell, then open **Answer** to check yourself. Most answers also carry a short written part; try it
in a markdown cell of your own before reading ours.

Every dataset was generated for this paper with its defects planted, so each answer can be checked
against what was actually planted. The data files sit in
[`endsem/practice/data/`](https://github.com/iitm-da/da2402/tree/master/endsem/practice/data), with
`make_data.py`, which wrote them.

**Setup.** Run the next cell first. `fetch(name)` downloads a file once into a local `data/` folder
and returns its path, so every later cell reads from disk. In Colab, first run
`!pip install dash dash-bootstrap-components rapidfuzz`; the other libraries are already there.


In [ ]:
import os, re, json, sqlite3
import requests
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from sklearn.impute import KNNImputer
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

URL = "https://raw.githubusercontent.com/iitm-da/da2402/master/endsem/practice/data/"

def fetch(name):
    """Download a data file once into ./data/ and return its local path."""
    path = os.path.join("data", name)
    if not os.path.exists(path):
        os.makedirs("data", exist_ok=True)
        r = requests.get(URL + name)
        r.raise_for_status()
        with open(path, "wb") as fh:
            fh.write(r.content)
    return path

### Q1  pandas · a merge that changes the row count

`orders.csv` holds 1,200 orders and `customers.csv` maps `customer_id` to `city`.

1. Inner-merge the two on `customer_id` and compare the row count with `len(orders)`.
2. Find the `customer_id` that appears twice in `customers`, and count the orders whose
   `customer_id` is missing from `customers` (the orphan orders) and their total `amount`.
3. Fix the merge: keep the **last** row for the duplicated customer (they moved city), use a left
   merge with `validate="many_to_one"`, and fill the orphans' city with `"Unknown"`.
4. Pivot revenue by `city` (rows) and quarter of `order_date` (columns). Report the city with the
   highest total and its Q4 revenue.

Answer variable `q1`: a dict with keys `orders`, `inner_merge_rows`, `duplicated_customer`, `orphan_orders`, `orphan_revenue`, `top_city`, `top_city_Q4`.

In [ ]:
q1 = ...   # your answer
q1

<details>
<summary><b>Answer</b></summary>

```python
orders = pd.read_csv(fetch("orders.csv"))
customers = pd.read_csv(fetch("customers.csv"))

inner = orders.merge(customers, on="customer_id", how="inner")
dup_id = customers.loc[customers.customer_id.duplicated(), "customer_id"].item()
orphans = ~orders.customer_id.isin(customers.customer_id)

# fix: keep the customer's later row, keep every order, label the unknowns
fixed = orders.merge(customers.drop_duplicates("customer_id", keep="last"),
                     on="customer_id", how="left", validate="many_to_one")
fixed["city"] = fixed["city"].fillna("Unknown")
fixed["quarter"] = pd.to_datetime(fixed["order_date"]).dt.quarter
pivot = fixed.pivot_table(index="city", columns="quarter", values="amount", aggfunc="sum")

q1 = {"orders": len(orders),
      "inner_merge_rows": len(inner),
      "duplicated_customer": dup_id,
      "orphan_orders": int(orphans.sum()),
      "orphan_revenue": round(float(orders.loc[orphans, "amount"].sum()), 2),
      "top_city": pivot.sum(axis=1).idxmax(),
      "top_city_Q4": round(float(pivot.loc[pivot.sum(axis=1).idxmax(), 4]), 2)}
q1
```

```
{'orders': 1200, 'inner_merge_rows': 1202, 'duplicated_customer': 42, 'orphan_orders': 9, 'orphan_revenue': 6518.69, 'top_city': 'Chennai', 'top_city_Q4': 73436.91}
```

**Written part.** The inner merge returned 1,202 rows, more than the 1,200 orders. Two things happened at once:
the 9 orphan orders were dropped, and customer 42's 11 orders were each matched to both of their
city rows, adding 11. The count went up by 2, which hides both defects. `validate="many_to_one"`
raises an error on the duplicated key instead, and that is why it belongs on every lookup merge.

</details>

### Q2  Regular expressions · delivery messages

`messages.txt` mixes delivery confirmations with OTPs, cancellations, a promotion and a
survey request. The confirmations come in three layouts, the currency is written `Rs.`, `Rs`,
`INR` or `₹`, amounts may carry thousands commas, and dates are `dd/mm/yyyy` or `dd-Mon-yyyy`.

1. Keep only lines that mention a delivery **and** carry an order id `#` + one capital letter +
   four digits.
2. From each, extract `order_id`, `date`, `amount` (a float) and the six-digit `pincode`.
3. Parse the dates with `pd.to_datetime(..., format="mixed", dayfirst=True)`.

Answer variable `q2`: a dict with `deliveries` (row count), `total_amount`, `by_month` (deliveries per month name, as a dict) and `top_city_prefix` (the most common first three digits of the pincode).

In [ ]:
q2 = ...   # your answer
q2

<details>
<summary><b>Answer</b></summary>

```python
lines = open(fetch("messages.txt"), encoding="utf-8").read().splitlines()

ORDER = re.compile(r"#([A-Z]\d{4})\b")
DATE = re.compile(r"\b(\d{2}[/-](?:\d{2}|[A-Z][a-z]{2})[/-]\d{4})\b")
AMOUNT = re.compile(r"(?:Rs\.?|INR|₹)\s?([\d,]+\.\d{2})")
PIN = re.compile(r"\b(\d{6})\b")

rows = []
for line in lines:
    if not (re.search(r"deliver", line, re.I) and ORDER.search(line)):
        continue                     # OTPs, cancellations, the promo and the survey
    rows.append({"order_id": ORDER.search(line).group(1),
                 "date": DATE.search(line).group(1),
                 "amount": float(AMOUNT.search(line).group(1).replace(",", "")),
                 "pincode": PIN.search(line).group(1)})
deliv = pd.DataFrame(rows)
deliv["date"] = pd.to_datetime(deliv["date"], format="mixed", dayfirst=True)

q2 = {"deliveries": len(deliv),
      "total_amount": round(float(deliv["amount"].sum()), 2),
      "by_month": deliv["date"].dt.month_name().value_counts().to_dict(),
      "top_city_prefix": deliv["pincode"].str[:3].value_counts().idxmax()}
q2
```

```
{'deliveries': 60, 'total_amount': 78948.09, 'by_month': {'August': 21, 'October': 20, 'September': 19}, 'top_city_prefix': '682'}
```

**Written part.** "Rate your delivery on 12/10/2026 and win Rs. 500" mentions a delivery and has a date, so a
filter on the word alone keeps it; it has no order id, which is what removes it. The OTP lines have
a six-digit number that the pincode pattern would happily match, which is why the line filter
comes before extraction. The amount pattern needs the comma inside the character class: 20 of the 60
amounts are written like `Rs. 1,870.23`, and `\d+\.\d{2}` does not match them at all.

</details>

### Q3  Scraping · pagination and a sponsored duplicate

`shop_p1.html` is page 1 of a product listing. Each product is a `div.card` with a `data-id`,
an `h3.title`, a `span.price` such as `₹1,299` and a `span.rating` such as `4.2 ★`. Out-of-stock
cards have no price span at all. Each page links to the next with `a.next`; the last page has none.

1. Start at page 1 and follow `a.next` until it runs out. Do not hard-code the page count.
2. Parse every card into a row with `id`, `title`, `price` (NaN when out of stock), `rating` and
   whether the card's class list contains `sponsored`.
3. Drop duplicates on `id`, keeping the first sighting.

Answer variable `q3`: a tuple `(cards_seen, unique_products, out_of_stock, mean_price)`, mean rounded to 2 dp.

In [ ]:
q3 = ...   # your answer
q3

<details>
<summary><b>Answer</b></summary>

```python
from bs4 import BeautifulSoup

page, rows = "shop_p1.html", []
while page:
    soup = BeautifulSoup(open(fetch(page), encoding="utf-8").read(), "html.parser")
    for card in soup.select("div.card"):
        price = card.select_one("span.price")
        rows.append({
            "id": card["data-id"],
            "title": card.select_one("h3.title").get_text(strip=True),
            "price": float(price.get_text().strip("₹").replace(",", "")) if price else np.nan,
            "rating": float(card.select_one("span.rating").get_text().split()[0]),
            "sponsored": "sponsored" in card["class"],
            "page": page})
    nxt = soup.select_one("a.next")
    page = nxt["href"] if nxt else None

shop = pd.DataFrame(rows)
products = shop.drop_duplicates("id")      # first sighting is the organic card

q3 = (len(shop), len(products), int(products["price"].isna().sum()),
      round(float(products["price"].mean()), 2))
q3
```

```
(34, 32, 2, 1550.4)
```

**Written part.** Pages 2 and 3 each open or close with a sponsored card that repeats a product listed
earlier, so 34 cards describe 32 products. Deduplicating on the title would be wrong here: the
`data-id` is the site's own key, and two products can share a title. A loop on `a.next` keeps
working when the shop adds a fourth page; `range(1, 4)` silently stops at three.

</details>

### Q4  APIs · flattening nested JSON

`weather.json` is a saved API response: a list of `stations`, each with an `id`, a `name`, a
nested `location` and a list of `readings`. Some readings have no `rain_mm` key.

1. Flatten it with `pd.json_normalize`, one row per reading, carrying the station's `id`, `name`
   and `location.lat` as metadata.
2. Per station name, report the number of readings, mean `temp_c`, number of readings with no
   rain value, and total rain.

Answer variable `q4`: a DataFrame indexed by station name with columns `readings`, `mean_temp`, `rain_missing`, `rain_total`, rounded to 2 dp.

In [ ]:
q4 = ...   # your answer
q4

<details>
<summary><b>Answer</b></summary>

```python
doc = json.load(open(fetch("weather.json"), encoding="utf-8"))
w = pd.json_normalize(doc["stations"], record_path="readings",
                      meta=["id", "name", ["location", "lat"]])

q4 = (w.groupby("name")
       .agg(readings=("time", "size"),
            mean_temp=("temp_c", "mean"),
            rain_missing=("rain_mm", lambda s: int(s.isna().sum())),
            rain_total=("rain_mm", "sum"))
       .round(2))
q4
```

```
          readings  mean_temp  rain_missing  rain_total
name                                                   
Adyar            8      28.09             0        22.4
Ennore           5      29.00             1        14.1
Guindy           8      28.45             0        23.9
Tambaram         8      26.66             3        17.0
```

**Written part.** `rain_total` is not comparable across stations. Ennore sent 5 readings because the station went
offline, and `sum()` skips the 3 missing rain values at Tambaram as though no rain fell. A missing
key in an API response is a missing value; it is zero rain only if the API documents it that way.

</details>

### Q5  Missing values · testing MCAR

`clinic.csv` has 500 patients. `age`, `bmi` and `sbp` are complete; `cholesterol` and
`sleep_hours` have missing values. For each of the two incomplete columns:

1. Run the permutation test on `age`: the statistic is |mean age of rows with the value missing −
   mean age of the rest|, 5,000 shuffles of the missingness indicator, one
   `np.random.default_rng(0)` per column, p = (1 + count of shuffled statistics ≥ observed) / (1 + 5000).
2. Fit a logistic regression of the missingness indicator on `age`, `bmi`, `sbp` with statsmodels
   and report the likelihood-ratio χ² and its p-value (`llr`, `llr_pvalue`).

Answer variable `q5`: a DataFrame with rows `cholesterol`, `sleep_hours` and columns `missing`, `perm_diff_age`, `perm_p`, `LR_chi2`, `LR_p`.

In [ ]:
q5 = ...   # your answer
q5

<details>
<summary><b>Answer</b></summary>

```python
clinic = pd.read_csv(fetch("clinic.csv"))

def perm_test(df, col, by, n_permutations=5000, seed=0):
    rng = np.random.default_rng(seed)
    m = df[col].isna().to_numpy()
    x = df[by].to_numpy()
    observed = abs(x[m].mean() - x[~m].mean())
    count = 0
    for _ in range(n_permutations):
        mp = rng.permutation(m)
        count += abs(x[mp].mean() - x[~mp].mean()) >= observed
    return observed, (1 + count) / (1 + n_permutations)

def lr_test(df, col, predictors):
    y = df[col].isna().astype(int)
    fit = sm.Logit(y, sm.add_constant(df[predictors])).fit(disp=0)
    return fit.llr, fit.llr_pvalue

table = {}
for col in ["cholesterol", "sleep_hours"]:
    diff, p = perm_test(clinic, col, "age")
    chi2, lr_p = lr_test(clinic, col, ["age", "bmi", "sbp"])
    table[col] = {"missing": int(clinic[col].isna().sum()), "perm_diff_age": round(diff, 3),
                  "perm_p": round(p, 4), "LR_chi2": round(chi2, 2), "LR_p": float(f"{lr_p:.3g}")}
q5 = pd.DataFrame(table).T
q5
```

```
             missing  perm_diff_age  perm_p  LR_chi2          LR_p
cholesterol    142.0         12.882  0.0002    74.16  5.470000e-16
sleep_hours     48.0          1.990  0.3979     1.68  6.420000e-01
```

**Written part.** Which column rejects MCAR, and what does the rejection leave open?

`cholesterol` rejects MCAR on both tests: patients missing it are 12.9 years older on average
(p = 0.0002), and the logistic's LR p is 5.5e-16. `sleep_hours` clears both (p = 0.40, 0.64). Both
tests share one null, MCAR. Rejecting it leaves MAR and MNAR standing together; no test separates
them, because MNAR depends on the value that was never recorded. Clearing it shows only that
missingness is unrelated to the columns tested.

</details>

### Q6  Imputation · scored against the truth

`clinic_truth.csv` holds `cholesterol` before any value was deleted. Fill the missing
`cholesterol` values three ways:

1. **mean** of the observed values;
2. **regression** on `age` and `bmi`, fitted on the complete rows;
3. **KNN** with `KNNImputer(n_neighbors=5)` on `age`, `bmi`, `sbp`, `cholesterol`, standardised
   first (subtract the mean, divide by the sd) and transformed back.

For the truth and each filled column, report RMSE on the filled rows only, mean, sd and correlation
with `age`.

Answer variable `q6`: a DataFrame with rows `truth`, `mean`, `regression`, `knn` and columns `rmse_filled`, `mean`, `sd`, `corr_age`, rounded to 3 dp.

In [ ]:
q6 = ...   # your answer
q6

<details>
<summary><b>Answer</b></summary>

```python
truth = pd.read_csv(fetch("clinic_truth.csv"))["cholesterol"]
miss = clinic["cholesterol"].isna()

fills = {"mean": clinic["cholesterol"].fillna(clinic["cholesterol"].mean())}

obs = clinic[~miss]
reg = LinearRegression().fit(obs[["age", "bmi"]], obs["cholesterol"])
r = clinic["cholesterol"].copy()
r[miss] = reg.predict(clinic.loc[miss, ["age", "bmi"]])
fills["regression"] = r

cols = ["age", "bmi", "sbp", "cholesterol"]
mu, sd = clinic[cols].mean(), clinic[cols].std()
k = KNNImputer(n_neighbors=5).fit_transform((clinic[cols] - mu) / sd)
fills["knn"] = pd.Series(k[:, 3] * sd["cholesterol"] + mu["cholesterol"])

def describe(v):
    return {"rmse_filled": np.sqrt(((v[miss] - truth[miss]) ** 2).mean()),
            "mean": v.mean(), "sd": v.std(), "corr_age": v.corr(clinic["age"])}

q6 = pd.DataFrame({"truth": describe(truth),
                   **{name: describe(v) for name, v in fills.items()}}).T.round(3)
q6.loc["truth", "rmse_filled"] = 0.0
q6
```

```
            rmse_filled     mean      sd  corr_age
truth             0.000  191.896  32.605     0.682
mean             35.754  186.364  27.018     0.546
regression       21.218  191.405  30.113     0.724
knn              24.871  190.662  30.196     0.705
```

**Written part.** Which fill would you use, and what is still wrong with it?

Regression has the lowest RMSE (21.2 against 35.8 for the mean). The mean fill shrinks the sd from
32.6 to 27.0 and drags the correlation with age from 0.682 to 0.546, and its mean is 5.5 low:
cholesterol is MAR on age, so the observed values come from younger patients. Regression puts every
filled point on the fitted line, so the correlation overshoots to 0.724 and the sd is still short
(30.1). Any model that fills with its prediction removes the scatter around it.

</details>

### Q7  Outliers · three univariate rules

`delivery_times.csv` has 300 delivery times in minutes, including a batch of orders that
stalled at the hub. Flag outliers in `minutes` three ways:

1. z-score, |z| > 3 (sample mean and sd);
2. IQR rule, outside [Q1 − 1.5·IQR, Q3 + 1.5·IQR];
3. modified z-score, 0.6745·(x − median)/MAD, |value| > 3.5.

Also list the values above the median that the IQR rule flags and the z-score misses.

Answer variable `q7`: a dict with the flag count per rule, `mean, sd` and the list of values z misses.

In [ ]:
q7 = ...   # your answer
q7

<details>
<summary><b>Answer</b></summary>

```python
t = pd.read_csv(fetch("delivery_times.csv"))
x = t["minutes"]

z = (x - x.mean()) / x.std()
Q1, Q3 = x.quantile([0.25, 0.75])
IQR = Q3 - Q1
med = x.median()
mad = (x - med).abs().median()
mz = 0.6745 * (x - med) / mad

flag = {"z > 3": z.abs() > 3,
        "IQR 1.5": (x < Q1 - 1.5 * IQR) | (x > Q3 + 1.5 * IQR),
        "modified z > 3.5": mz.abs() > 3.5}
q7 = {name: int(f.sum()) for name, f in flag.items()}
q7["mean, sd"] = (round(float(x.mean()), 2), round(float(x.std()), 2))
q7["z misses (slow side)"] = sorted(x[flag["IQR 1.5"] & ~flag["z > 3"] & (x > med)].tolist())
q7
```

```
{'z > 3': 13, 'IQR 1.5': 20, 'modified z > 3.5': 19, 'mean, sd': (37.13, 23.23), 'z misses (slow side)': [54.0, 89.9, 90.3, 92.6, 97.9, 105.0]}
```

**Written part.** Why does the z-score miss values the other two catch?

Masking. The 18 stalled orders pull the mean up to 37.1 and the sd up to 23.2 (the normal batch
has a sd near 6), so a 90-minute delivery sits under 3 sds from the mean. The median and IQR barely
move when 6% of the data is extreme, so the other two rules see it. The IQR rule's extra flags
include 54.0 at the slow end and 14.7 at the fast end, ordinary orders from the normal batch: a
flag says a value is unusual, and deciding whether it is wrong takes knowing how it was produced.

</details>

### Q8  Outliers · Mahalanobis against LOF

`sensors.csv` has temperature and vibration readings from two machines that run at
different settings, with three planted outliers among them.

1. Compute each reading's squared Mahalanobis distance from the overall mean using the overall
   covariance, and flag readings above `stats.chi2.ppf(0.975, df=2)`.
2. Standardise the two columns and fit `LocalOutlierFactor(n_neighbors=20)`. Take the 5 readings
   with the lowest `negative_outlier_factor_`.

Answer variable `q8`: a dict with `mahalanobis` (sorted reading ids flagged) and `lof_top5` (ids, most outlying first).

In [ ]:
q8 = ...   # your answer
q8

<details>
<summary><b>Answer</b></summary>

```python
s = pd.read_csv(fetch("sensors.csv"))
X = s[["temp_c", "vibration_mm_s"]].to_numpy()

diff = X - X.mean(axis=0)
d2 = np.einsum("ij,jk,ik->i", diff, np.linalg.inv(np.cov(X, rowvar=False)), diff)
maha = s.loc[d2 > stats.chi2.ppf(0.975, df=2), "reading_id"].tolist()

lof = LocalOutlierFactor(n_neighbors=20).fit(StandardScaler().fit_transform(X))
lof_top5 = s["reading_id"].to_numpy()[np.argsort(lof.negative_outlier_factor_)[:5]].tolist()

q8 = {"mahalanobis": sorted(maha), "lof_top5": lof_top5}
q8
```

```
{'mahalanobis': [41, 131, 194, 240, 241, 266, 270], 'lof_top5': [143, 240, 241, 80, 27]}
```

**Written part.** The planted readings are 143 (just off machine A's tight cloud), 113 (between the machines)
and 240 (far from both). Why does Mahalanobis miss 143 and 113?

One mean and one covariance describe a single elliptical cloud. With two machines, the overall mean
falls in the empty space between them: reading 113 sits almost on it (d² = 0.01). Machine B's
spread inflates the covariance, so 143 at d² = 0.41 looks ordinary, while normal points at B's edge
get flagged instead. LOF compares each point with its own neighbours, so 143 has the largest
factor of all. 113 escapes LOF too (rank 12): with only a few points nearby, it reads as sparse
rather than isolated. Neither method knows which machine a reading came from; fitting one per
machine is the fix when that label exists.

</details>

### Q9  Duplicates · normalise, block, match

`members.csv` has sign-ups from one club, with people who registered more than once. Their
names vary in case, spacing, a `Dr.` prefix, a one-letter typo or a `Surname, First` order; their
phone numbers vary as `9840012345`, `+91 98400 12345`, `+91-9840012345` or `09840012345`. A few
signed up again with a new number. `members_truth.csv` maps each `record_id` to a `person_id`.

1. Clean names (lower case, collapse spaces, drop `dr.`, put `surname, first` back in order) and
   reduce each phone to its last 10 digits.
2. Block on `city`: form candidate pairs only within a city, each pair once.
3. Score name similarity with `rapidfuzz.distance.JaroWinkler.similarity`.
4. For each rule, count predicted pairs, true positives, precision and recall: raw phone equal;
   clean phone equal; name JW ≥ 0.93; clean phone equal **or** name JW ≥ 0.97.

Answer variable `q9`: a tuple `(all_pairs, pairs_after_blocking, true_pairs, table)` with the rules as rows.

In [ ]:
q9 = ...   # your answer
q9

<details>
<summary><b>Answer</b></summary>

```python
from rapidfuzz.distance import JaroWinkler

mem = pd.read_csv(fetch("members.csv"), dtype={"phone": str})
truth9 = pd.read_csv(fetch("members_truth.csv"))

def clean_name(n):
    n = re.sub(r"\s+", " ", n.lower().strip())
    n = re.sub(r"^dr\.?\s+", "", n)
    if "," in n:                                   # "iyer, priya" -> "priya iyer"
        last, first = [p.strip() for p in n.split(",", 1)]
        n = f"{first} {last}"
    return n

mem["name_c"] = mem["name"].map(clean_name)
mem["phone10"] = mem["phone"].str.replace(r"\D", "", regex=True).str[-10:]
mem = mem.merge(truth9, on="record_id")

n = len(mem)
pairs = mem.merge(mem, on="city", suffixes=("_a", "_b"))          # block on city
pairs = pairs[pairs["record_id_a"] < pairs["record_id_b"]].copy()
pairs["jw"] = [JaroWinkler.similarity(a, b) for a, b in zip(pairs["name_c_a"], pairs["name_c_b"])]
pairs["same"] = pairs["person_id_a"] == pairs["person_id_b"]
true_pairs = int((mem.groupby("person_id").size().apply(lambda k: k * (k - 1) // 2)).sum())

rules = {"raw phone equal": pairs["phone_a"] == pairs["phone_b"],
         "clean phone equal": pairs["phone10_a"] == pairs["phone10_b"],
         "name jw >= 0.93": pairs["jw"] >= 0.93,
         "clean phone or name jw >= 0.97": (pairs["phone10_a"] == pairs["phone10_b"]) | (pairs["jw"] >= 0.97)}
res = {}
for name, hit in rules.items():
    tp = int((hit & pairs["same"]).sum())
    res[name] = {"predicted": int(hit.sum()), "true_pos": tp,
                 "precision": round(tp / hit.sum(), 3), "recall": round(tp / true_pairs, 3)}
q9 = (n * (n - 1) // 2, len(pairs), true_pairs, pd.DataFrame(res).T)
q9
```

```
(30135, 7474, 105,                                 predicted  true_pos  precision  recall
raw phone equal                       5.0       5.0      1.000   0.048
clean phone equal                    92.0      92.0      1.000   0.876
name jw >= 0.93                     125.0     105.0      0.840   1.000
clean phone or name jw >= 0.97      110.0     104.0      0.945   0.990)
```

**Written part.** Blocking on city cuts 30,135 pairs to 7,474 and loses no true pair, since a person's copies
share a city here. Comparing raw phones finds 5 of 105 true pairs; cleaning them finds 92, with no
false match, and misses the people with a new number. Names alone find all 105 but bring 20 false
pairs: 5 are different people with exactly the same name, and most of the rest share a first name,
which Jaro-Winkler rewards because it weights the common prefix (`karthik raman` against
`karthik nair` scores 0.934). The combined rule trades a little of each:
precision 0.945, recall 0.990. Every threshold here was chosen with the truth file open; on real
data you would label a sample of pairs to choose it.

</details>

### Q10  Storage · what a CSV forgets

`meters.csv` and `meters.parquet` hold the same 400 meter readings. Read both with pandas
defaults and compare the dtypes. Add a row showing the first `meter_id` from each file as a
string.

Answer variable `q10`: a DataFrame with columns `csv`, `parquet`, one row per column plus the `first meter_id` row.

In [ ]:
q10 = ...   # your answer
q10

<details>
<summary><b>Answer</b></summary>

```python
a = pd.read_csv(fetch("meters.csv"))
b = pd.read_parquet(fetch("meters.parquet"))

q10 = pd.DataFrame({"csv": a.dtypes.astype(str), "parquet": b.dtypes.astype(str)})
q10.loc["first meter_id"] = [str(a["meter_id"].iloc[0]), str(b["meter_id"].iloc[0])]
q10
```

```
                    csv         parquet
meter_id          int64          object
read_on          object  datetime64[ns]
zone             object        category
units           float64         float64
first meter_id    20770          020770
```

**Written part.** CSV stores text and no types, so pandas guesses: `meter_id` `020770` becomes the integer
20770, the date stays a string and the zone becomes object. Parquet stores the schema with the data,
so the leading zero, the datetime and the category come back as written. To read the CSV
correctly, say so: `pd.read_csv(..., dtype={"meter_id": str}, parse_dates=["read_on"])`.

</details>

### Q11  Storage · SQLite joins and anti-joins

`shop.db` has tables `customers` (customer_id, name, city), `products` (product_id,
product, category, price) and `orders` (order_id, customer_id, product_id, qty, order_date).
Write SQL and run it with `pd.read_sql`:

1. Revenue (`qty × price`) and order count per city, highest revenue first.
2. Customers who never ordered, with a `LEFT JOIN … WHERE o.order_id IS NULL`.
3. Products never sold, with `NOT EXISTS`.

Answer variable `q11`: a dict with `top_city`, `top_revenue` (int), `never_ordered` (list of ids) and `unsold` (list of product names).

In [ ]:
q11 = ...   # your answer
q11

<details>
<summary><b>Answer</b></summary>

```python
con = sqlite3.connect(fetch("shop.db"))

by_city = pd.read_sql("""
    SELECT c.city, COUNT(*) AS orders, SUM(o.qty * p.price) AS revenue
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    JOIN products  p ON p.product_id  = o.product_id
    GROUP BY c.city
    ORDER BY revenue DESC""", con)

never = pd.read_sql("""
    SELECT c.customer_id FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    WHERE o.order_id IS NULL ORDER BY c.customer_id""", con)

unsold = pd.read_sql("""
    SELECT p.product FROM products p
    WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.product_id = p.product_id)""", con)

q11 = {"top_city": by_city.loc[0, "city"], "top_revenue": int(by_city.loc[0, "revenue"]),
       "never_ordered": never["customer_id"].tolist(), "unsold": unsold["product"].tolist()}
q11
```

```
{'top_city': 'Pune', 'top_revenue': 731216, 'never_ordered': [7, 23, 51, 64, 78], 'unsold': ['Item L', 'Item Q']}
```

**Written part.** An inner join of customers to orders drops the 5 customers who never ordered without any
warning, which is why the anti-join needs the LEFT JOIN and the `IS NULL` test on a column from
the right-hand table.

</details>

### Q12  Dash · a callback you can test without a browser

Use Gapminder (`px.data.gapminder()`, bundled with plotly).

1. Lay the app out with `dash_bootstrap_components`: a row with a 4-wide column holding a
   continent `dcc.Dropdown` (id `continent`, default `"Asia"`) and an `html.H4` (id `kpi`) under
   it, and an 8-wide column holding a `dcc.Graph` (id `trend`).
2. Write one callback from `continent` to both outputs: the KPI text
   `"<continent>: median life expectancy <x.x> years in 2007"` and a line chart of median
   `lifeExp` per year.
3. A decorated callback is still a plain Python function. Call it with `"Africa"` and inspect what
   it returns. Run `app.run(jupyter_mode="inline")` only to look at the app.

Answer variable `q12`: a tuple `(kpi_text, first three y values rounded to 1 dp, number of x points)` from `update("Africa")`.

In [ ]:
# build app, write update(continent), then:
q12 = ...   # your answer
q12

<details>
<summary><b>Answer</b></summary>

```python
from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc
import plotly.express as px

gm = px.data.gapminder()
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
app.layout = dbc.Container([
    dbc.Row([
        dbc.Col([dcc.Dropdown(id="continent", options=sorted(gm["continent"].unique()),
                              value="Asia", clearable=False),
                 html.H4(id="kpi", className="mt-3")], md=4),
        dbc.Col(dcc.Graph(id="trend"), md=8)])
], fluid=True)

@app.callback(Output("kpi", "children"), Output("trend", "figure"),
              Input("continent", "value"))
def update(continent):
    med = (gm[gm["continent"] == continent]
           .groupby("year", as_index=False)["lifeExp"].median())
    kpi = f"{continent}: median life expectancy {med['lifeExp'].iloc[-1]:.1f} years in 2007"
    fig = px.line(med, x="year", y="lifeExp", markers=True,
                  title=f"Median life expectancy, {continent}")
    return kpi, fig

# app.run(jupyter_mode="inline")      # uncomment to see the app
kpi, fig = update("Africa")
q12 = (kpi, [round(float(v), 1) for v in fig.data[0].y[:3]], len(fig.data[0].x))
q12
```

```
('Africa: median life expectancy 52.9 years in 2007', [38.8, 40.6, 42.6], 12)
```

**Written part.** Which gestalt principle tells a reader that the dropdown controls the KPI and not just the
chart? Proximity and common region: the dropdown and the KPI share one column, so they read as a
group, and the chart sits in its own. Moving the KPI into the chart's column would make it read as
the chart's title. Testing the callback as a function catches a wrong median or a wrong label
without starting a server.

</details>

### Q13  Honest charts · lie factor and per capita

`enrolment.csv` has made-up college enrolment and population figures, in lakh, for five
states in 2024 and 2025.

1. A news graphic drew Tamil Nadu's 2024 and 2025 enrolment as bars on an axis starting at 10.0.
   Compute the change the bars show (the change in bar height above 10.0), the actual change, and
   the lie factor (shown ÷ actual).
2. For 2025, compute enrolment per 1,000 people and draw an honest horizontal bar chart sorted by
   it, axis from zero (plotly's default for bars).
3. Report which state leads by raw count and which leads per 1,000.

Answer variable `q13`: a dict with `shown_change_%`, `actual_change_%`, `lie_factor`, `top_by_count`, `top_per_1000` and `per_1000` (state → value).

In [ ]:
q13 = ...   # your answer
q13

<details>
<summary><b>Answer</b></summary>

```python
e = pd.read_csv(fetch("enrolment.csv"))
tn = e[e["state"] == "Tamil Nadu"].set_index("year")["enrolled_lakh"]

baseline = 10.0                                  # where the published axis started
shown = (tn[2025] - baseline) / (tn[2024] - baseline) - 1
actual = tn[2025] / tn[2024] - 1

e25 = e[e["year"] == 2025].assign(
    per_1000=lambda d: (d["enrolled_lakh"] / d["population_lakh"] * 1000).round(1))
honest = px.bar(e25.sort_values("per_1000"), x="per_1000", y="state", orientation="h",
                title="Enrolled per 1,000 people, 2025")

q13 = {"shown_change_%": round(float(100 * shown), 1),
       "actual_change_%": round(float(100 * actual), 1),
       "lie_factor": round(float(shown / actual), 1),
       "top_by_count": e25.loc[e25["enrolled_lakh"].idxmax(), "state"],
       "top_per_1000": e25.loc[e25["per_1000"].idxmax(), "state"],
       "per_1000": dict(zip(e25["state"], e25["per_1000"]))}
q13
```

```
{'shown_change_%': 195.2, 'actual_change_%': 4.0, 'lie_factor': 48.6, 'top_by_count': 'Tamil Nadu', 'top_per_1000': 'Goa', 'per_1000': {'Tamil Nadu': 13.7, 'Kerala': 12.7, 'Karnataka': 13.3, 'Telangana': 16.4, 'Goa': 21.2}}
```

**Written part.** A 4.0% rise is drawn as a 195% rise, a lie factor of 48.6. Tamil Nadu leads on raw count only
because it is the most populous of the five; per 1,000 people, Goa leads at 21.2. Goa's 0.34 lakh
is the smallest count in the table, so its rate moves most from one year's noise, and a
choropleth of the raw counts would paint Tamil Nadu darkest and say nothing about enrolment. For a
per-1,000 map, use a sequential scale: one hue from light to dark, since the quantity runs one way
from low to high.

</details>

That is the paper. Errors you spot in it: tell Dr. Ayyar in class or on Moodle.